# Exercise 6 · A flaky test: fix the cause, do not retry

A test passes alone and fails in the full run. You find the cause, and Codex fixes it.

**Goal:** Find why a test fails only in some runs, and have Codex fix the cause.

**Why it matters:** A test that fails only sometimes makes every red result doubtful. A retry can turn a flaky test green and keeps the cause.

**Before you start:**
- You installed Codex and signed in (course README, step 1).

**What the check measures:** the tests pass in file order, in reverse order and one at a time. The asserts stay the same, and nothing retries or skips. You committed the fix in `notes.py`. Your answer to one question is right.

**Done when:** the last cell prints `Done:`.

## The situation

`notes.py` adds a note to a case. `test_notes.py` has two tests. Each passes alone. In the full run, one fails. The files hold made-up data.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It writes `practice/` as it is after Step 6: `notes.py` with Codex's fix, the unchanged `test_notes.py`, and two commits. It never overwrites a file.

Expect three `Wrote practice/` lines and `Committed the start and the fix`.

In [ ]:
FILES = {
    "notes.py": """\"\"\"Notes that staff add to a case.\"\"\"


def add_note(case_id: str, note: str, notes: list[dict] | None = None) -> list[dict]:
    \"\"\"Add a note for a case to a list of notes.

    Args:
        case_id: The case the note is for, such as C-1.
        note: The text of the note.
        notes: The list to add the note to.

    Returns:
        The list of notes, with the new note last.
    \"\"\"
    if notes is None:
        notes = []
    notes.append({"case": case_id, "note": note})
    return notes
""",
    "test_notes.py": """from notes import add_note


def test_first_note() -> None:
    notes = add_note("C-1", "Called the taxpayer")
    assert len(notes) == 1


def test_two_notes() -> None:
    notes = add_note("C-2", "Sent a letter")
    notes = add_note("C-2", "The letter came back", notes)
    assert len(notes) == 2
""",
    ".gitignore": "__pycache__/\n.pytest_cache/\n",
}

from pathlib import Path

PRACTICE = Path("practice")
PRACTICE.mkdir(exist_ok=True)
for name, text in FILES.items():
    if (PRACTICE / name).exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        (PRACTICE / name).write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

# Codex's fix in Step 5 (simulated): each call with no list makes a new list.
import subprocess

if not (PRACTICE / ".git").exists():
    fixed = FILES["notes.py"]
    start = fixed.replace("list[dict] | None = None", "list[dict] = []").replace("    if notes is None:\n        notes = []\n", "")
    (PRACTICE / "notes.py").write_text(start, encoding="utf-8")
    for step in (["init", "-q", "-b", "main"], ["add", "-A"], ["commit", "-qm", "Start"]):
        subprocess.run(["git", "-C", str(PRACTICE), *step], check=True)
    (PRACTICE / "notes.py").write_text(fixed, encoding="utf-8")
    subprocess.run(["git", "-C", str(PRACTICE), "commit", "-qam", "Make a new notes list on each call"], check=True)
    print("Committed the start and the fix")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-06-flaky-test-cause/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line.

## Step 3 · Run the tests in three ways

First all of them, then each one alone, then both in the other order:

```sh
pytest -q test_notes.py
pytest -q test_notes.py::test_first_note
pytest -q test_notes.py::test_two_notes
pytest -q test_notes.py::test_two_notes test_notes.py::test_first_note
```

Expect four results. The full run: `1 failed, 1 passed`, with `assert 3 == 2` in `test_two_notes`. Each one alone: `1 passed`. The other order: `test_first_note` fails with `assert 3 == 1`. The test that runs second fails. Read the `where 3 = len([...])` line: the list holds the other test's note.

### Learn: a test that depends on the order

"Properly isolated tests can be run in any sequence." ([Fowler](https://martinfowler.com/articles/nonDeterminism.html)). In a study of 7,571 flaky tests in Python projects, order dependency caused 59% ([Gruber and others](https://arxiv.org/abs/2101.09077)). In real work the order changes when you run one test alone or add a test. Parallel runs change it too ([pytest](https://docs.pytest.org/en/stable/explanation/flaky.html)).

## Step 4 · Name the cause

Why did the second test fail only after the first?

- **A.** Both tests used one list: Python makes the default `[]` once.
- **B.** pytest runs the tests in a random order.
- **C.** The second test ran too fast.

Write your letter between the quotes, then run the cell. Expect `Saved your answer: ` and your letter.

<details><summary>Show the answer</summary>

A. Why: in both orders, the second test saw the first test's note. Python makes the default list once, so every call with no list adds to it ([Python tutorial](https://docs.python.org/3/tutorial/controlflow.html#default-argument-values)).

</details>

In [ ]:
ANSWER = "A"
print(f"Saved your answer: {ANSWER}")

## Step 5 · Ask Codex for the cause and a fix

```sh
codex --no-daemon
```

If Codex asks "Trust this folder?", choose `1. Trust and continue`. Type this request and press Enter:

```text
test_notes.py passes one test at a time and fails in the full run. Find the cause and fix it there. Do not add a retry or a skip. Do not change any assert. Done when pytest passes in file order, in reverse order and one test at a time.
```

Expect Codex to name the default list `= []` in `notes.py` and change it.

## Step 6 · Check the fix, then commit it

Type `/diff` and read the change. Press `q`, then type `/exit`. Run the four Step 3 lines again. Expect `2 passed`, `1 passed`, `1 passed`, `2 passed`. Keep the fix only if they pass and `/diff` shows no change to `test_notes.py`. Otherwise run `git restore .` and go back to Step 5. Commit:

```sh
git commit -qam "Make a new notes list on each call"
```

<details><summary>Show the answer</summary>

A fix at the cause: the default becomes `None`, then `if notes is None: notes = []`. Each call with no list then gets a new list.

</details>

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It writes the Step 1 files again. Set it back to `False` and go to Step 3.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. Expect five `ok:` lines, then `Done:`.

In [ ]:
# Check your work: read the tests and notes.py, run pytest three ways in a copy of practice/, read Git and your answer
import ast
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

PRACTICE = Path("practice")
ASSERTS = {"test_first_note": ["len(notes) == 1"], "test_two_notes": ["len(notes) == 2"]}


def failed_runs(tests: list) -> list:
    """Return the runs that fail, in a copy of practice/."""
    runs = {"file order": [], "reverse order": [f"test_notes.py::{t}" for t in reversed(tests)]}
    runs.update({f"{t} alone": [f"test_notes.py::{t}"] for t in tests})
    with tempfile.TemporaryDirectory() as tmp:
        copy = shutil.copytree(PRACTICE, Path(tmp) / "p", ignore=shutil.ignore_patterns(".git", "__pycache__"))
        return [name for name, args in runs.items() if subprocess.run(
            [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *(args or ["test_notes.py"])],
            cwd=copy, capture_output=True).returncode]


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    test_file, notes_file = PRACTICE / "test_notes.py", PRACTICE / "notes.py"
    if not test_file.is_file() or not notes_file.is_file():
        return "practice/notes.py or practice/test_notes.py is missing. Run the Step 1 cell."
    source = test_file.read_text(encoding="utf-8")
    tests = {node.name: [ast.unparse(a.test) for a in ast.walk(node) if isinstance(a, ast.Assert)]
             for node in ast.parse(source).body if isinstance(node, ast.FunctionDef)}
    if any(tests.get(name) != asserts for name, asserts in ASSERTS.items()) or any(
            word in source.lower() for word in ("rerun", "flaky", "skip")):
        return "test_notes.py has a changed assert, a retry or a skip. Run git restore test_notes.py (Step 6)."
    print("ok: the asserts are unchanged, with no retry and no skip")
    failed = failed_runs(list(ASSERTS))
    if failed:
        return f"the tests fail in this run: {', '.join(failed)}. Ask Codex for the cause and a fix (Step 5)."
    print("ok: the tests pass in file order, in reverse order and one at a time")
    add = next(n for n in ast.parse(notes_file.read_text(encoding="utf-8")).body if getattr(n, "name", "") == "add_note")
    if any(isinstance(default, (ast.List, ast.Dict, ast.Set)) for default in add.args.defaults):
        return "add_note in notes.py still has a list as a default. Fix it there (Step 5)."
    print("ok: add_note has no shared default list")
    git = ["git", "-C", str(PRACTICE)]
    log = subprocess.run([*git, "log", "--format=%s"], capture_output=True, text=True).stdout
    if (not (PRACTICE / ".git").is_dir() or len(log.splitlines()) < 2
            or subprocess.run([*git, "status", "--porcelain"], capture_output=True).stdout):
        return "the fix is not committed. Commit it (Step 6)."
    print("ok: the fix is committed")
    if str(globals().get("ANSWER", "")).strip().upper() != "A":
        return "the question has no answer, or not the right one. Answer it again (Step 4)."
    print("ok: your answer is right")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run the Step 1 cell."
print(f"Not yet: {problem}" if problem else "Done: the tests pass in any order, and nothing retries or skips.")

## What you learned

- A test that passes alone and fails in the full run depends on the order.
- Fix the cause, here a shared default list. A retry or a skip keeps the cause and hides the next real failure.

**Reflect:** Which tests in your own work share data that one test changes?